# **Use Python to match U.S. Census population and housing permit data in 5 year increments.

# Goal: Be able to make connections to which cities will be impacted by the Road to Housing act

1. **building permits**, 2016- 2020 and 2021–2025, from the Census Building Permits Survey

4. **population**, from Census population estimates

**How I matched the data sets:** I used the Census location code instead of the town names to avoid differences in spelling and format.

Data will be stored in `analysis/wharton_permits_matched_2021_2025.csv`

## Setup

In [43]:
import pandas as pd

## Step 2: Load the Census building permits, 2021–2025
https://www2.census.gov/econ/bps/Place/ 

The Census Building Permits Survey publishes text files of how many housing permits every permit-issuing place issued
Important to note:
1. **Includes number of units per building:** 1 unit, 2 units, 3–4 units, 5+ units
2. **Published every year**
3. **Divided by region:** Midwest, Northeast, South, West
4. **"Months reported" = 0:** Means the town didn't submit data that year, not that it was actually 0.

In [44]:
# Retrieve all 20 regional permitting data sets from Census
regions = ["mw", "ne", "so", "we"]    # Midwest, Northeast, South, West
years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

files = []
for region in regions:
    for year in years:
        path = f"../raw_data/census/bps/{region}{year}a.txt"
        files.append(pd.read_csv(path, skiprows=3, header=None, dtype=str, encoding="latin-1")) #the file has two header lines and one blank line at the top this code makes sure these are skipped so it can run. It also prompts pandas to make numbered columns and turns everything into text.

# Combine 20 data sets into a new data frame, clean it and print its size
bps = pd.concat(files, ignore_index=True)
bps = bps.apply(lambda col: col.str.strip()) #Removes padding spaces at the beginning and end of town names.
print(bps.shape)

(200637, 41)


In [45]:
# Keep only these columns (the Census file gives a key for number meanings)
bps = bps[[0, 1, 5, 6, 15, 16, 18, 21, 24, 27]]

# Make columns have simpler names
bps = bps.rename(columns={
    0: "year",
    1: "state_code",
    5: "place_code",
    6: "cousub_code",
    15: "months_reported",
    16: "bps_name",
    18: "units_1",
    21: "units_2",
    24: "units_3_4",
    27: "units_5plus",
})

num_cols = ["months_reported", "units_1", "units_2", "units_3_4", "units_5plus"]
bps[num_cols] = bps[num_cols].astype(int)

In [46]:
# Create columns for total new homes and apartments
# (Note: apartments are classified as any building with 2 or more residential units)
bps["units_all"] = bps["units_1"] + bps["units_2"] + bps["units_3_4"] + bps["units_5plus"]
bps["units_apartments"] = bps["units_all"] - bps["units_1"]

# Remove all rows where munipalities didn't submit data
# 0 means missing data not 0 homes built
print(f"share of rows not reported: {(bps['months_reported'] == 0).mean():.3f}") #Asked Claude for help with this f string
bps = bps[bps["months_reported"] > 0]
bps.head()

share of rows not reported: 0.179


,year,state_code,place_code,cousub_code,months_reported,bps_name,units_1,units_2,units_3_4,units_5plus,units_all,units_apartments
0,2016,17,00113,11891,12,Abingdon,0,0,0,0,0,0
1,2016,17,00243,00250,12,Addison village,3,0,0,0,3,0
2,2016,17,00516,00529,12,Albany village,0,0,0,0,0,0
3,2016,17,00555,44667,12,Albers village,1,0,0,0,1,0
4,2016,17,00646,48398,12,Aledo,0,0,0,0,0,0


## Step 3: Edit the way the location code is written so they can be matched

**The Permit data set does not have the same 7-digit code so I had to combine the state code with the local code.**
There are two types of places recorded:
- **Cities:** `place_code`
- **Towns/townships:** `cousub_code`

In [47]:
# Build the 7-digit code so population and permits data can be matched
bps["match_key"] = bps["state_code"] + bps["place_code"]
is_township = bps["place_code"] == "00000"
bps.loc[is_township, "match_key"] = bps["state_code"] + bps["cousub_code"]

# Drop county-level rows
county_rows = bps["place_code"].str.startswith("99") | (is_township & (bps["cousub_code"] == "00000"))
matched = bps[~county_rows]
print("permit rows kept:", len(matched))


permit rows kept: 149329


In [48]:
#Create an old (2016-2020) and new (2021-2025) data frame

new = matched[matched["year"].astype(int) >= 2021]
old = matched[matched["year"].astype(int) <= 2020]

# Add up each town's permits over all years, and count how many years it reported
permits = new.groupby("match_key").agg({
    "units_1": "sum",
    "units_apartments": "sum",
    "units_all": "sum",
    "year": "nunique",
    "bps_name": "first",
}).reset_index().rename(columns={"year": "years_reported"})

# Divide the totals by the number of years with data to get the average per year
permits["units_1_avg_2021_25"] = permits["units_1"] / permits["years_reported"]
permits["units_apartments_avg_2021_25"] = permits["units_apartments"] / permits["years_reported"]
permits["units_all_avg_2021_25"] = permits["units_all"] / permits["years_reported"]

# Average permits per year for 2016-2020 (add up each year first, so NYC's five boroughs count as one place)
old_avg = old.groupby(["match_key", "year"])["units_all"].sum().groupby("match_key").mean().reset_index(name="units_all_avg_2016_20")
permits = pd.merge(permits, old_avg, on="match_key", how="left")

# Exclude places without at least 3 years of permit data
permits = permits[permits["years_reported"] >= 3]
print("places with 3+ years of permits:", len(permits))

places with 3+ years of permits: 15209


## Step 4: Population
https://www2.census.gov/programs-surveys/popest/datasets/

**This data requires the same procedure as the permit data to get the 7-digit location code**
Download the data sets needs for 2018-2022 and to measure growth before and after
- 2010s file: 2010 census count, 2017–2019 estimates
- 2020s file: 2020–2023 estimates

In [49]:
# Load the population file (downloaded from Census into data/census/)
pop_2020s = pd.read_csv("../raw_data/census/sub-est2025.csv", encoding="latin-1", dtype=str)

In [50]:
# Create the 7-digit code to match to the other data set by combining the town/city and state codes.
places = pop_2020s[pop_2020s["SUMLEV"] == "162"]
places["match_key"] = places["STATE"] + places["PLACE"]

cousubs = pop_2020s[pop_2020s["SUMLEV"] == "061"]
cousubs["match_key"] = cousubs["STATE"] + cousubs["COUSUB"]

pop_2020s = pd.concat([places, cousubs])

In [51]:
# Remove duplicates (same 7-digit code listed twice) and keep only the population columns needed
pop_2020s = pop_2020s.groupby("match_key")[["STNAME", "POPESTIMATE2021", "POPESTIMATE2022",
                                            "POPESTIMATE2023", "POPESTIMATE2024","POPESTIMATE2025"]].first()

# Don't count cells that list A instead of a number (only the population columns, not the state name)
pop_cols = ["POPESTIMATE2021", "POPESTIMATE2022", "POPESTIMATE2023", "POPESTIMATE2024", "POPESTIMATE2025"]
pop_2020s[pop_cols] = pop_2020s[pop_cols].replace("A", None).astype(float)
pop_2020s = pop_2020s.reset_index()
pop_2020s.head()

,match_key,STNAME,POPESTIMATE2021,POPESTIMATE2022,POPESTIMATE2023,POPESTIMATE2024,POPESTIMATE2025
0,0100124,Alabama,2340.0,2338.0,2351.0,2391.0,2378.0
1,0100460,Alabama,4329.0,4264.0,4219.0,4208.0,4178.0
2,0100484,Alabama,665.0,669.0,666.0,672.0,669.0
3,0100676,Alabama,228.0,226.0,231.0,234.0,236.0
4,0100820,Alabama,33732.0,33890.0,34177.0,34463.0,34748.0


## Step 5: Time to actually combine everything!!!!

In [52]:
# Match permits data and population data based on the 7-digit code
#Inner drops places without matches
towns_2025 = pd.merge(permits, pop_2020s, on="match_key", how="inner")

In [53]:
#Create a column for population averages.
towns_2025["pop_avg_2021_25"] = towns_2025[["POPESTIMATE2021", "POPESTIMATE2022", "POPESTIMATE2023",
                                            "POPESTIMATE2024", "POPESTIMATE2025"]].mean(axis=1)


#Create column for permits granted per 1,000 based on census population data.
towns_2025["permits_per_thousand_2025"]= (towns_2025["units_all_avg_2021_25"] / towns_2025["pop_avg_2021_25"]) * 1000

towns_2025

,match_key,units_1,units_apartments,units_all,years_reported,bps_name,units_1_avg_2021_25,units_apartments_avg_2021_25,units_all_avg_2021_25,units_all_avg_2016_20,STNAME,POPESTIMATE2021,POPESTIMATE2022,POPESTIMATE2023,POPESTIMATE2024,POPESTIMATE2025,pop_avg_2021_25,permits_per_thousand_2025
0,0100124,41,0,41,4,Abbeville,10.25,0.0,10.25,0.25,Alabama,2340.0,2338.0,2351.0,2391.0,2378.0,2359.6,4.343957
1,0100460,0,0,0,3,Adamsville,0.00,0.0,0.00,0.00,Alabama,4329.0,4264.0,4219.0,4208.0,4178.0,4239.6,0.000000
2,0100820,451,0,451,5,Alabaster,90.20,0.0,90.20,97.60,Alabama,33732.0,33890.0,34177.0,34463.0,34748.0,34202.0,2.637273
3,0100988,287,8,295,5,Albertville,57.40,1.6,59.00,27.60,Alabama,22524.0,22788.0,23159.0,23522.0,23710.0,23140.6,2.549631
4,0101132,271,56,327,5,Alexander City,54.20,11.2,65.40,16.80,Alabama,14700.0,14615.0,14503.0,14498.0,14515.0,14566.2,4.489846
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15187,5681300,0,0,0,5,Wamsutter town,0.00,0.0,0.00,0.00,Wyoming,199.0,197.0,198.0,197.0,197.0,197.6,0.000000
15188,5683040,34,0,34,5,Wheatland town,6.80,0.0,6.80,11.00,Wyoming,3598.0,3578.0,3538.0,3501.0,3516.0,3546.2,1.917546
15189,5684925,12,2,14,5,Worland,2.40,0.4,2.80,1.40,Wyoming,4813.0,4820.0,4814.0,4756.0,4730.0,4786.6,0.584966
15190,5685015,2,0,2,5,Wright town,0.40,0.0,0.40,0.20,Wyoming,1620.0,1631.0,1643.0,1658.0,1659.0,1642.2,0.243576


In [54]:
#Drop the places that don't have permits per capita

towns_2025 = towns_2025.dropna(subset=["permits_per_thousand_2025"])
towns_2025

,match_key,units_1,units_apartments,units_all,years_reported,bps_name,units_1_avg_2021_25,units_apartments_avg_2021_25,units_all_avg_2021_25,units_all_avg_2016_20,STNAME,POPESTIMATE2021,POPESTIMATE2022,POPESTIMATE2023,POPESTIMATE2024,POPESTIMATE2025,pop_avg_2021_25,permits_per_thousand_2025
0,0100124,41,0,41,4,Abbeville,10.25,0.0,10.25,0.25,Alabama,2340.0,2338.0,2351.0,2391.0,2378.0,2359.6,4.343957
1,0100460,0,0,0,3,Adamsville,0.00,0.0,0.00,0.00,Alabama,4329.0,4264.0,4219.0,4208.0,4178.0,4239.6,0.000000
2,0100820,451,0,451,5,Alabaster,90.20,0.0,90.20,97.60,Alabama,33732.0,33890.0,34177.0,34463.0,34748.0,34202.0,2.637273
3,0100988,287,8,295,5,Albertville,57.40,1.6,59.00,27.60,Alabama,22524.0,22788.0,23159.0,23522.0,23710.0,23140.6,2.549631
4,0101132,271,56,327,5,Alexander City,54.20,11.2,65.40,16.80,Alabama,14700.0,14615.0,14503.0,14498.0,14515.0,14566.2,4.489846
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15187,5681300,0,0,0,5,Wamsutter town,0.00,0.0,0.00,0.00,Wyoming,199.0,197.0,198.0,197.0,197.0,197.6,0.000000
15188,5683040,34,0,34,5,Wheatland town,6.80,0.0,6.80,11.00,Wyoming,3598.0,3578.0,3538.0,3501.0,3516.0,3546.2,1.917546
15189,5684925,12,2,14,5,Worland,2.40,0.4,2.80,1.40,Wyoming,4813.0,4820.0,4814.0,4756.0,4730.0,4786.6,0.584966
15190,5685015,2,0,2,5,Wright town,0.40,0.0,0.40,0.20,Wyoming,1620.0,1631.0,1643.0,1658.0,1659.0,1642.2,0.243576


## Step 6: Save

In [55]:
towns_2025.to_csv("../analysis/wharton_permits_matched_2021_2025.csv", index=False)
print("saved", len(towns_2025), "towns_2025")

saved 15192 towns_2025
